# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and perform basic analysis on the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined using the Croissant schema at:

[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Install the mlcroissant library, if necessary
!pip install mlcroissant

## 1. Data Loading
Load metadata and inspect dataset-level information.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")


## 2. Data Overview
Let's review available record sets and their structure according to the Croissant schema.

We will enumerate all record sets, their `@id`s, and their fields. All references use the entity `@id` as per the dataset schema.

In [ ]:
# List all record sets in the dataset
record_sets = []
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    # Some versions may use 'recordSet' instead
    record_sets = getattr(metadata, 'recordSet', [])

print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    fields = rs.get('field', [])
    # Ensure fields is a list
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields ({len(fields)}):")
    for field in fields:
        field_id = field.get('@id') if isinstance(field, dict) else field
        print(f"    - Field @id: {field_id}")
    print()
if not record_sets:
    print('No explicit record sets found in top-level metadata. Checking for default datasets in mlcroissant...')

# For this dataset and mlcroissant, the primary record set is usually `cr:RecordSet-1`
# Let's try to enumerate by using the dataset's available record sets through the API:
rs_ids = dataset.record_sets()
print(f"RecordSet @id(s) available in this dataset: {rs_ids}")


## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis.

Below, each record set is loaded separately, referenced by its `@id`.

*Use the Record Set `@id` shown above as a reference for data loading.*

In [ ]:
# Extract data from each record set
# Get the available record set @ids
record_set_ids = dataset.record_sets()

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for Record Set @id: {record_set_id}")
    try:
        # Some record sets may have a lot of records; limit to first 5000 just in case
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded DataFrame shape: {df.shape}")
        print(f"  Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"  Error loading record set {record_set_id}: {e}")
    print()
# Show the first record set's dataframe head for reference
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"First 5 records in main record set ({main_rs_id}):")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's perform basic EDA: filter records, normalize values in a numeric field, and group by a relevant field.

Edit the cell below to select the appropriate field `@id`s for numeric analysis.

In [ ]:
import numpy as np

# Select the main record set and a numeric field for filtering and normalization
main_record_set_id = list(dataframes.keys())[0]

df = dataframes[main_record_set_id]

# Print available fields
print(f"Available fields (@id/column names): {df.columns.tolist()}")

# Pick one numeric field -- e.g., 'cr:field-MolecularWeight' or a similar ID
# You may wish to change this if you know the correct @id from the dataset structure
numeric_field = None
candidate_numeric_fields = [col for col in df.columns if 'weight' in col.lower() or 'abundance' in col.lower() or 'coverage' in col.lower() or 'count' in col.lower()]
if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
else:
    numeric_field = df.columns[0]  # fallback
print(f"Using numeric field: {numeric_field}")

# Convert field to numeric if not already
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Outlier filtering: filter for values greater than a threshold
threshold = np.nanpercentile(df[numeric_field], 25)  # 25th percentile as demo threshold
filtered_df = df[df[numeric_field] > threshold]
print(f"Records with {numeric_field} > {threshold:.2f}:")
print(filtered_df[[numeric_field]].head())

# Normalize the field (standard score)
mean = filtered_df[numeric_field].mean()
std = filtered_df[numeric_field].std()
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Pick a candidate group (categorical) field, e.g., by 'cr:field-Accession' or description field
categorical_candidates = [col for col in df.columns if 'description' in col.lower() or 'accession' in col.lower() or 'sample' in col.lower()]
group_field = categorical_candidates[0] if categorical_candidates else None

if group_field:
    print(f"Grouping by {group_field}...")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
    print(grouped_df.head())
else:
    print("No suitable group field for grouping found.")

## 5. Visualization
Let's visualize data distributions and relationships in the main record set. Histograms and scatter plots are often informative for numeric fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Histogram for the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If we have a group field, show a boxplot
if group_field and df[group_field].nunique() <= 20:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- The FAIR^2 dataset provides detailed mass spectrometry-based measurements annotated with rich metadata and field structure via the Croissant schema.
- We demonstrated metadata exploration, field and record set inspection, basic EDA, and simple visualizations using `mlcroissant` and pandas.
- All schema entities and columns are referenced and handled using their Croissant `@id` fields for reproducibility.

For further data analysis, consult the [Croissant specification](https://mlcommons.org/croissant), the domain documentation, or extend this notebook for domain-specific processing!